First, create the model. This must match the model used in the interactive training notebook.

In [1]:
from jetcam.csi_camera import CSICamera

from jetracer.nvidia_racecar import NvidiaRacecar

import cv2

import torch

import torchvision

CATEGORIES = ['apex']

device = torch.device('cuda')
model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2 * len(CATEGORIES))
model = model.cuda().eval().half()

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


Next, load the saved model.  Enter the model path you used to save.

In [2]:
model.load_state_dict(torch.load('road_following_wide_model.pth'))

<All keys matched successfully>

Convert and optimize the model using ``torch2trt`` for faster inference with TensorRT.  Please see the [torch2trt](https://github.com/NVIDIA-AI-IOT/torch2trt) readme for more details.

> This optimization process can take a couple minutes to complete. 

In [3]:
from torch2trt import torch2trt

data = torch.zeros((1, 3, 126, 224)).cuda().half()
model_trt = torch2trt(model, [data], fp16_mode=True)

Save the optimized model using the cell below

In [4]:
torch.save(
    model_trt.state_dict(),
    'road_following_wide_trt.pth'
)

Load the optimized model by executing the cell below

In [1]:

from jetcam.csi_camera import CSICamera

from jetracer.nvidia_racecar import NvidiaRacecar

import cv2

import torch

import torchvision
from torch2trt import torch2trt

from torch2trt import TRTModule

model_trt = TRTModule()

model_trt.load_state_dict(
    torch.load('road_following_wide_trt.pth')
)

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


<All keys matched successfully>

Create the racecar class

In [2]:
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()

Create the camera class.

In [3]:
from jetcam.csi_camera import CSICamera

camera = CSICamera(
    width=640,
    height=360,
    capture_width=1280,
    capture_height=720,
    capture_fps=30
)

camera.running = False

In [4]:
frame = camera.read()

print("Camera shape:", frame.shape)

Camera shape: (360, 640, 3)


In [5]:
from utils import preprocess

image = preprocess(frame).half()

print("Model input:", image.shape)

Model input: torch.Size([1, 3, 360, 640])


In [6]:
car.steering_offset = 0.07

In [7]:
car.steering_gain = -0.9

Finally, execute the cell below to make the racecar move forward, steering the racecar based on the x value of the apex.

Here are some tips,

* If the car wobbles left and right,  lower the steering gain
* If the car misses turns,  raise the steering gain
* If the car tends right, make the steering bias more negative (in small increments like -0.05)
* If the car tends left, make the steering bias more postive (in small increments +0.05)

In [12]:
from utils import preprocess
import numpy as np

STEERING_GAIN = 1
STEERING_BIAS = 0.00

car.throttle = 0.12

while True:
    image = camera.read()
    image = preprocess(image).half()
    output = model_trt(image).detach().cpu().numpy().flatten()
    x = float(output[0])
    car.steering = x * STEERING_GAIN + STEERING_BIAS

KeyboardInterrupt: 

# Final Task: Interactive JetRacer Control and Calibration and keep inside the track
Create an interactive Jupyter Notebook widget using ipywidgets to control and calibrate the JetRacer.

The widget should include a live camera preview and sliders to adjust steering gain, steering bias, and throttle in real time.

Find solution how to keep always jetracer inside track. if it is out of the track or mat then stop it.

Add separate Start/Stop Camera and Start/Stop Car buttons to control the camera and vehicle independently.

